In [100]:
from bs4 import BeautifulSoup
import numpy as np
import re, os, email, json, numbers, pickle

# Parsing document

## read

### read source

In [101]:
data_root = '/home/jhyang/WORKSPACES/DATA/ICSD_articles'
paths = []
soups = []
htmls = []
for pub in os.listdir(os.path.join(data_root, 'src')):
    path_pub = os.path.join(data_root, 'src', pub)
    for n in os.listdir(path_pub):
        path = os.path.join(path_pub, n)
        if os.path.isfile(path):
            jn = ''
            paths.append([pub, jn, path])
        else:
            jn = n
            for fn in os.listdir(path):
                paths.append([pub, jn, os.path.join(path, fn)])
for pub, jn, path_fn in paths:
    with open(path_fn, 'r', encoding='utf-8') as f:
        if path_fn.endswith('.mhtml'):
            msg = email.message_from_file(f)
            for part in msg.walk():
                if part.get_content_type() == 'text/html':
                    break
            html = part.get_payload(decode=True).decode('utf-8')
        else:
            html = f.read()
        soup = BeautifulSoup(html, 'html.parser')
        soups.append(soup)
        htmls.append(html)

### DOI 

In [102]:
doi_full_pattern = re.compile(r'(?:https?://doi\.org/|doi:)\s*(10\.\d{4,9}/[-._;()/:A-Z0-9]+)', re.IGNORECASE)

def parse_doi(text):
    match = doi_full_pattern.search(text)
    if match:
        return match.group(1)
    else:
        return text

In [103]:
dois = []
for i, (soup, (pub, jn, html)) in enumerate(zip(soups, paths)):
    _dois = []
    for meta in soup.find_all('meta'):
        if meta.get('name') is None:
            continue
        if 'doi' in meta.get('name'):
            _dois.append(parse_doi(meta.get('content')))
    _dois = np.unique(_dois)
    if len(_dois) == 1:
        dois.append(_dois[0])
    else:
        print(i, pub, jn, html)
        [print(doi) for doi in _dois]
        # check
    

### synthesis paragraph

In [ ]:
count = []
hist = {'abs':[], 'intro':[], 'conc':[], 'ref':[], 'ack':[]}
for soup in soups:
    headers = soup.find_all('h1 h2 h3 h4 h5 h6'.split())
    c = [False, False, False, False, False]
    for i, header in enumerate(headers):

        text = header.get_text().lower()
        if 'abstract' in text:
            c[0] = True
            hist['abs'].append(text)
        if 'intro' in text:
            c[1] = True
            hist['intro'].append(text)
        if 'conclu' in text or 'outlook' in text:
            c[2] = True
            hist['conc'].append(text)
        if 'reference' in text:
            c[3] = True
            hist['ref'].append(text)
        if 'acknowledg' in text:
            c[4] = True
            hist['ack'].append(text)
    count.append(c)
count = np.array(count)

In [104]:
def get_main_text(soup):
    doc = {}
    collect = False
    removed = []
    for i, tag in enumerate(soup.find_all('h1 h2 h3 h4 h5 h6 p div'.split())):
        if tag.find('h1 h2 h3 h4 h5 h6 p div'.split()) is not None:
            continue
        text = tag.get_text().strip()
        if len(text) == 0:
            continue
        if tag.name.startswith('h'):
            if any([x in text.lower() for x in ['intro','abstract','keyword']]):
                collect = True
            if any([text.lower().startswith(x) for x in ['refere','acknowledg','credit','appendix','conflict']]):
                break
            if any([x in text.lower() for x in ['intro','keyword','reference','license','author','supporting','interest','supplement']]) or any([text.lower().startswith(x) for x in ['fig','table']]):
                header = None
            elif collect and ((i, text) not in doc.keys()):
                header = (i, text)
                doc[header] = []
            if (not collect) or (header is None):
                removed.append((i, text))
            continue
#        print(collect, header, text)
        if not collect: 
            continue
        if header is None:
            continue
        doc[header].append((i, text))
    return {k:v for k,v in doc.items() if len(v) != 0}, removed

def print_document(doc, chunk=200):
#    blacklist = ['intro','abstract','conclu','acknowledg','reference','license','author','supporting','table','figure','keyword','interest']
    whitelist = ['synthe','prep','heat','anneal','pressed','cool']
    for (i, header), paragraph in doc.items():
#        if any([x in header.lower() for x in blacklist]):
#            continue
#        if len(paragraph) == 0:
#            continue
        print('='*chunk)
        print('{:>7s}\t{}'.format(f'({i})', header))
        print('-'*chunk)
        for (j, sentence) in paragraph:
            sentence = sentence.replace('\n',' ')
            for k in range(0, len(sentence), chunk):
                s = sentence[k:k+chunk]
                for x in whitelist:
                    s = s.replace(x,f'\x1b[43m\x1b[30m{x}\x1b[0m')
                if k == 0:
                    print('{:6d} \t{}'.format(j, s))
                else:
                    print('       \t{}'.format(s))
    for i in range(5): print()
    print('==     E   N   D     =='.center(chunk))
    for i in range(5): print()

def update_documents(documents, doi, soup, idxs):
    if isinstance(idxs, numbers.Integral):
        _idxs = [idxs]
    elif isinstance(idxs, str):
        _idxs = [int(l) for l in idxs.replace(',',' ').strip().split()]
    elif isinstance(idxs, (list, tuple)):
        _idxs = [int(l) for l in idxs]
    elements = soup.find_all('h1 h2 h3 h4 h5 h6 p div'.split())
    documents[doi] = [elements[idx].get_text() for idx in sorted(_idxs)]
    return documents

def print_soup(tag, level=0):
    print('  ' * level, f'<{tag.name}>', tag.text.strip().replace('\n','<br>'))
    for child in tag.children:
        if child.name:
            print_soup(child, level + 1)

In [276]:
docs = {}
for i, (doi, soup) in enumerate(zip(dois, soups)):
    doc, _ = get_main_text(soup)
    docs[doi] = doc
    if len(doc) == 0:
        print(i, doi)

with open(os.path.join(data_root, 'parsed_documents.pkl'),'wb') as f:
    pickle.dump(docs, f)

185 10.5194/ejm-35-373-2023
226 10.1107/S2414314621007355
227 10.1107/S2056989021000633
229 10.1107/S2414314621009883


In [271]:
i = 185
doi = dois[i]
doc, _ = get_main_text(soups[i])
for k, v in doc.items():
    print(k, len(v))
print('=')
for k, v in docs_[doi].items():
    print(k, len(v))

=


In [273]:
soup = soups[185]#print_soup(soups[181])
print_context = True
for i, tag in enumerate(soup.find_all('h1 h2 h3 h4 h5 h6'.split())):
#    if tag.find('h1 h2 h3 h4 h5 h6 p div'.split()) is not None:
#        continue
#    if tag.name != 'p':
#        if 'abstract' in tag.get_text().lower():
#            print_context = True
#        elif 'acknowledge' in tag.get_text().lower():
#            print_context = False
    if print_context:
        print(i, tag.name, tag.text.strip().replace('\n','<b>'))

0 h1 Article
1 h1 hits for
2 h1 Network problems
3 h1 Server timeout
4 h1 Empty search term
5 h1 Too many requests
6 h1 Fe-bearing vanadium dioxide–paramontroseite:  structural details and high-temperature transformation
7 h3 Nadia Curetti
8 h3 Alessandro Pavese
9 h2 2.1 Sample
10 h2 2.2 Chemical maps and analyses
11 h2 2.3 Single-crystal X-ray diffraction
12 h2 2.4 In situ high-temperature X-ray powder diffraction
13 h2 2.5 Raman spectroscopy on single crystals
14 h2 3.1 Single-crystal X-ray diffraction
15 h2 3.2 In situ high-temperature X-ray powder diffraction
16 h2 3.3 Raman spectroscopy on natural and heated crystals


## find synthesis paragraphs

In [105]:
with open('../dump/documents_synthesis_only.json','r') as f:
    documents = json.load(f)
processed_dois = [dois.index(k) for k in documents.keys()]
m = np.ones_like(dois, dtype=bool)
m[processed_dois] = False
m[np.max(processed_dois):] = False
len(documents), np.max(processed_dois), np.where(m)[0]

(109, 117, array([ 5, 30, 33, 48, 56, 62, 69, 90, 94]))

In [ ]:
i = 117
doc, removed = get_main_text(soups[i])
print(f'https://doi.org/{dois[i]}')
print_document(doc, chunk=190)

In [301]:
idxs = '84 101 162'
documents = update_documents(documents, dois[i], soups[i], idxs)

with open('../dump/documents_synthesis_only.json','w') as f:
    json.dump(documents, f)

In [ ]:
eles = soups[i].find_all('h1 h2 h3 h4 h5 h6 p div'.split())
print(eles[125])

In [ ]:
chunk = 150
for doi, para in documents.items():
    print(doi)
    for sentence in para:
        for i in range(0, len(sentence), chunk):
            print(f'\t{sentence[i:i+chunk]}')

In [ ]:
documents[dois[i]] = 

In [ ]:
html = htmls[0]
#html = html.replace('>','>\n')
while '\n\n' in html:
    html = html.replace('\n\n','\n')
lines = [l for l in html.split('\n') if len(l.strip()) != 0]
for i,l in enumerate(lines):
    if 'intro' in l.lower():
        print('='*50)
    print(f'{i}\t{l}')


In [75]:
example = 'test conclusion'
any([x in example for x in ['conclu','refere','acknowledge','outlook','discuss']])

True

# LLM APIs

## groq

In [2]:
import os
from groq import Groq

api_root = os.path.expanduser('~/.api_keys')
api_key = open(os.path.join(api_root, 'groq')).read().strip()
client = Groq(api_key=api_key)

In [49]:
def chat(msg, model='mixtral-8x7b-32768'):
    if isinstance(msg, str):
        messages = [{'role':'user', 'content':msg}]
    elif isinstance(msg, (list, tuple)):
        messages = [{'role':'user', 'content':_msg} for _msg in msg]
    else:
        raise ValueError('Invalid input for chat', type(msg))
    response = client.chat.completions.create(messages=messages, model=model)
    return response

def print_chat(choices, maxlen=95):
    for choice in choices:
        text = choice.message.content.split('\n')
        code = False
        for _text in text:
            if _text.startswith('```'):
                code = not code
            if code:
                print(_text)
            else:
                c = 0
                for token in _text.split():
                    c += len(token) + 1
                    if c > maxlen:
                        print()
#                        print(c, token)
                        c = len(token) + 1
                    print(token, end=' ')
                print()

In [50]:
message = chat(
   [
       'I will give you chunk of html documents that consisting single article. this includes various useless informations. I need text part that starts from intro or abstract to conclusion. all the other part such as title, metadata, supporting info, acknowledgements are not needed. give me the main part of the article in text format'
    ] + [
        htmls[0][i:i+5000] for i in range(0, len(htmls[0]), 5000)
   ],
)
print_chat(message.choices)

APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `mixtral-8x7b-32768` in organization `org_01j55mv9bdfxfvmjcgptqqrrb2` on tokens per minute (TPM): Limit 5000, Requested 168062, please reduce your message size and try again. Visit https://console.groq.com/docs/rate-limits for more information.', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

## langchain-groq

In [1]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

In [3]:
llm = ChatGroq(
    model='llama3-8b-8192',
    temperature=0.1,
    max_tokens=4096,
    api_key=api_key
)

prompt = ChatPromptTemplate.from_messages([
    
])

# Transformers

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from typing import List, Dict
import os

class TextBasedLLM:
    def __init__(self, model_path: str, device: str = "cuda" if torch.cuda.is_available() else "cpu"):
        """
        Args:
            model_path: 로컬에 저장된 모델 경로 (예: "meta-llama/Llama-3.2-1B")
            device: 사용할 디바이스 ("cuda" 또는 "cpu")
        """
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_path,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
            low_cpu_mem_usage=True
        ).to(device)
        
        self.conversation_history: List[Dict] = []
                    
    def generate_response(self, prompt: str, max_length: int = 2048) -> str:
        """응답 생성"""
        # 대화 기록을 포함한 전체 프롬프트 생성
        full_prompt = self._build_prompt(prompt)
        
        # 입력 인코딩
        inputs = self.tokenizer(full_prompt, return_tensors="pt").to(self.device)
        
        # 응답 생성
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_length=max_length,
                num_return_sequences=1,
                temperature=0.7,
                top_p=0.95,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id
            )
        
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # 프롬프트 부분 제거하여 실제 응답만 추출
        response = response[len(full_prompt):].strip()
        
        # 대화 기록 업데이트
        self.conversation_history.append({"role": "user", "content": prompt})
        self.conversation_history.append({"role": "assistant", "content": response})
        
        return response
    
    def _build_prompt(self, prompt: str) -> str:
        """대화 기록을 포함한 프롬프트 생성"""
        conversation = ""
        for message in self.conversation_history[-4:]:  # 최근 4개 메시지만 포함
            conversation += f"{message['role']}: {message['content']}\n"
        return f"{conversation}user: {prompt}\nassistant:"
    
    def start_conversation(self, context: str):
        """대화 시작"""

        # 시스템 프롬프트 설정
        system_prompt = (
            "You are a helpful AI assistant. You will be provided with a text document, "
            "and you should answer questions based on that document. "
            "If the answer cannot be found in the document, say so.\n\n"
            f"Document content:\n{context}\n"
        )
        self.conversation_history = [{"role": "system", "content": system_prompt}]
        
        print("대화를 시작합니다. 종료하려면 'quit' 또는 'exit'를 입력하세요.")
        
        while True:
            user_input = input("\nUser: ").strip()
            
            if user_input.lower() in ['quit', 'exit']:
                print("대화를 종료합니다.")
                break
                
            try:
                response = self.generate_response(user_input)
                print(f"\nAssistant: {response}")
            except Exception as e:
                print(f"Error: {e}")
                continue


class TokenOptimizedChatMemory:
    def __init__(self, model_name="gpt-3.5-turbo", max_memory_tokens=500):
        self.memory = []  # 대화 메모리를 저장하는 리스트
        self.max_memory_tokens = max_memory_tokens
        self.tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
        self.generator = pipeline("text-generation", model=model_name, device=0)
        self.embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')  # 임베딩 모델

    def _calculate_memory_tokens(self):
        """현재 메모리의 총 토큰 수 계산."""
        return sum(len(self.tokenizer.encode(item)) for item in self.memory)

    def _truncate_memory(self):
        """토큰 제한 초과 시, 중요도를 고려하여 메모리 최적화."""
        total_tokens = self._calculate_memory_tokens()

        if total_tokens <= self.max_memory_tokens:
            return

        # 임베딩 기반 중요도 계산
        embeddings = self.embedder.encode(self.memory)
        central_embedding = np.mean(embeddings, axis=0).reshape(1, -1)
        scores = cosine_similarity(embeddings, central_embedding).flatten()

        # 중요도가 낮은 순서대로 삭제
        sorted_indices = np.argsort(scores)  # 낮은 중요도 순서대로 정렬
        while total_tokens > self.max_memory_tokens and sorted_indices.size > 0:
            idx_to_remove = sorted_indices[0]
            self.memory.pop(idx_to_remove)
            sorted_indices = np.delete(sorted_indices, 0)  # 제거된 인덱스 업데이트
            total_tokens = self._calculate_memory_tokens()

    def extract_and_store_info(self, user_input):
        """사용자 입력에서 정보를 추출 및 저장."""
        summary_prompt = f"Summarize the key information from this input:\n{user_input}\nSummary:"
        summary = self.generator(summary_prompt, max_length=50, num_return_sequences=1)[0]['generated_text']
        self.memory.append(summary)
        self._truncate_memory()

    def update_memory(self, update):
        """새로운 정보를 추가 및 메모리 최적화."""
        self.memory.append(update)
        self._truncate_memory()

    def generate_response(self, user_input):
        """메모리를 기반으로 GPT-3 응답 생성."""
        memory_context = " ".join(self.memory)
        prompt = f"Memory:\n{memory_context}\nUser input:\n{user_input}\nResponse:"
        response = self.generator(prompt, max_length=150, num_return_sequences=1)[0]['generated_text']
        return response.strip()




In [305]:

MODEL_PATH = "meta-llama/Llama-3.2-1B"

llm = TextBasedLLM(MODEL_PATH)

TEXT_PATH = "path/to/your/text/file.txt"
    
llm.start_conversation()

<module 'llama_stack' from '/home/jhyang/anaconda3/envs/llm/lib/python3.9/site-packages/llama_stack/__init__.py'>

In [120]:
model_path = 'meta-llama/Llama-3.2-3B'
tokenizer = AutoTokenizer.from_pretrained(model_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
            model_path,
            low_cpu_mem_usage=True,
            output_hidden_states=True,
            pad_token_id=tokenizer.pad_token_id,
            max_length=4096,
        ).to('cuda')

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [152]:
prompt = '''
<START OF SYSTEM INSTRUCTION>
1. extract synthesis information from given TEXT. response must strictly adhere to the provided TEXT.
2. Information includes target materials' composition, types of precursors used, processing time and temperature.
3. target and precursors should be chemical composition such as TiO2 rather than titanium dioxide.
4. if target composition is given as range (a, b, c or x, y, z), find the best composition within context. if not, state the range.
5. time and temperature could be complexed (e.g, multiple heating & cooling cycles). In that case, give me representative condition with their unit.
6. if temperature and time is given as range, find the best temperature within context.
7. some article didn't describe any detail about synthesis condition. in that case, just collect as your best.
8. documents can includes multiple synthesis ways for either same or different target materials. state all the reaction seperatly.
9. original TEXT that includes information about the synthesis must be included for comparison.
10. The TEXT will given by chunk due to its long length. so, extract core informations from the TEXT to update memory.
11. if memory is given, update the new information from the TEXT.
12. Do not repeat content from the previous response unless explicitly requested.
13. Only the information from given TEXT must be included.
<END OF SYSTEM INSTRUCTION>

'''

def extract_information(text, memory=''):
    full_text = prompt
    full_text += f'<START OF MEMORY>\n{memory}\n<END OF MEMORY>' if len(memory) != 0 else ''
    full_text += f'<START OF TEXT>:\n{text}<END OF TEXT>\n\nInformation:\n'

    inp = tokenizer(full_text, return_tensors="pt").to('cuda')
    with torch.no_grad():
        out = model.generate(**inp,
                            repetition_penalty=1.2,  # 패널티 값 설정
                            no_repeat_ngram_size=2,)
    output_text = tokenizer.decode(out[0][0])
    return output_text#.replace(full_text,'')
#    

In [153]:
out = extract_information(doc[(82,'Abstract')][0][1])

/home/jhyang/anaconda3/envs/llm/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:774: UserWarning: `return_dict_in_generate` is NOT set to `True`, but `output_hidden_states` is. When `return_dict_in_generate` is not `True`, `output_hidden_states` is ignored.
  warnings.warn(


In [154]:
print(out)

<|begin_of_text|>
<START OF SYSTEM INSTRUCTION>
extract synthesis information from given TEXT. response must strictly adhere to the provided TEXT.
Information includes target materials' composition, types of precursors used, processing time and temperature.
target and precursors should be chemical composition such as TiO2 rather than titanium dioxide.
if target composition is given as range (a, b, c or x, y, z), find the best composition within context. if not, state the range.
time and temperature could be complexed (e.g, multiple heating & cooling cycles). In that case, give me representative condition with their unit.
if temperature and time is given as range, find the best temperature within context.
some article didn't describe any detail about synthesis condition. in that case, just collect as your best.
documents can includes multiple synthesis ways for either same or different target materials. state all the reaction seperatly.
original TEXT that includes information about the 

In [145]:
text = doc[(82,'Abstract')][0][1]
for i in range(0, len(text), 80):
    print(text[i:i+80])

A new solid solution of composition PbBiFe1-xMxO4 (M ​= ​Mn and Co, 0 ​≤ ​x ​≤ ​
0.15–0.2) with low structural dimensions has been synthesized by solid-state rea
ction at 873–923 ​K. The crystal structures have been investigated using X-ray a
nd neutron diffraction. The presence of Co2+/Co3+ on the Fe3+ sites and oxygen v
acancies in PbBiFe1-xCoxO4 is revealed by its unusual volume increase and TGA, w
hile the volume contraction of PbBiFe1-xMnxO4 indicates the substitution of Mn4+
/Mn3+ on the Fe3+ sites. This leads to unexpected greater and smaller metal-meta
l distances inside the chains for the Co and Mn case, respectively, compared wit
h the undoped materials. All the PbBiFe1-xMxO4 materials display long-range anti
ferromagnetic order but are suppressed by cationic substitution, as evidenced by
 a decrease in TN. Additional intrinsic short-range ferro- (or ferri-) magnetic 
transitions in the Co-doped materials are evidenced by magnetic hysteresis and f
requency-dependent χ’ peak a

In [ ]:
'''
Information:
- target materials: PbBiFe1-xMxO4
- precursors: PbO, Bi2O3, Fe2O3, MnO, CoO
- synthesis condition: solid-state reaction at 873–923 ​K
- information about synthesis: 
    - PbBiFe1-xCoxO4: Co2+/Co3+ on the Fe3+ sites and oxygen vacancies
    - PbBiFe1-xMnxO4: Mn4+/Mn3+ on the Fe3+ sites
    - all the PbBiFe1-xMxO4 materials display long-range antiferromagnetic order but are suppressed by cationic substitution, as evidenced by a decrease in TN
    - additional intrinsic short-range ferro- (or ferri-) magnetic transitions in the Co-doped materials are evidenced by magnetic hysteresis and frequency-dependent χ’ peak around TF
    - the introduction of ferromagnetic intrachain clusters in PbBiFe1-xCoxO4 is rationalized through a competitive mechanism between direct AFM exchange interaction and 90°- type FM superexchange interactions, which provided insights to tune the magnetic properties and design low dimensional functional materials.
- original text: A new solid solution of composition PbBiFe1-xMxO4 (M ​= ​Mn and Co, 0 ​≤ ​x ​≤ ​0.15–0.2) with low structural dimensions has been synthesized by solid-state reaction at 873–923 ​K. The crystal structures have been investigated using X-ray and neutron diffraction. The presence of Co2+/Co3+ on the Fe3+ sites and oxygen vacancies in PbBiFe1-xCoxO4 is revealed by its unusual volume increase and TGA, while the volume contraction of PbBiFe1-xMnxO4 indicates the substitution of Mn4+/Mn3+ on the Fe3+ sites. This leads to unexpected greater and smaller metal-metal distances inside the chains for the Co and Mn case, respectively, compared with the undoped materials. All the PbBiFe1-xMxO4 materials display long-range antiferromagnetic order but are suppressed by cationic substitution, as evidenced by a decrease in TN. Additional intrinsic short-range ferro- (or ferri-) magnetic transitions in the Co-doped materials are evidenced by magnetic hysteresis and frequency-dependent χ’ peak around TF. The introduction of ferromagnetic intrachain clusters in PbBiFe1-xCoxO4 is rationalized through a competitive mechanism between direct AFM exchange interaction and 90°- type FM superexchange interactions, which provided insights to tune the magnetic properties and design low dimensional functional materials.

 
'''